# Book Market Analytics – Books to Scrape
**Data Science 500 Assignment** · Richfield Graduate Institute of Technology

This notebook covers the whole project, from collecting the data to building a model. It follows the **five steps of data science**:

1. **Ask an interesting question**
2. **Obtain the data** (web scraping with BeautifulSoup)
3. **Explore the data** (cleaning and visualisation)
4. **Model the data** (linear regression)
5. **Communicate and visualise the results**

The work also uses all three parts of the Data Science Venn Diagram: programming skills (Python), maths and statistics (averages, correlation, regression), and domain knowledge (how books are sold and priced).

> **How to run it:** run the cells from top to bottom (*Kernel → Restart & Run All*). Scraping all 1,000 books takes about 5–10 minutes.

## Setup: import the libraries

In [ ]:
# Web scraping
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import re

# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make plots a bit bigger and easier to read
plt.rcParams["figure.figsize"] = (9, 5)
sns.set_style("whitegrid")

print("Libraries loaded")

## Step 1 – Ask an interesting question

A publishing analytics team wants to know what affects book prices and ratings on Books to Scrape. My questions are:

- How are book prices spread out, and what is a typical price?
- Do books with higher star ratings cost more?
- Do some genres cost more than others?
- Does stock availability have anything to do with price?
- **Can a linear regression model predict a book's price from information shown on the website?**

## Step 2 – Obtain the data (web scraping)

Books to Scrape (https://books.toscrape.com) is a practice website made for learning web scraping. It has 1,000 books split into 50 genres, with 20 books on each listing page.

My plan:
1. Check `robots.txt` to see if the site lists any rules for scrapers.
2. Read the genre links from the sidebar on the home page.
3. Go through every page of every genre by following the **next** button. From each book "card" I collect the **title, price, star rating, stock status and genre**.
4. Visit each book's own page to get the **number of copies in stock** (e.g. "In stock (22 available)"), the **UPC** code, the **number of reviews** and the **description**.

Scraping genre by genre means every book gets its genre label, which the main listing pages don't show.

### Settings

In [ ]:
BASE_URL = "https://books.toscrape.com/"

# A polite delay (in seconds) between requests so we don't overload the server
DELAY_SECONDS = 0.3

# Visiting each book's own page gives the stock quantity and the description.
# It takes about 5-10 minutes for 1,000 books. Set this to False for a quick run.
SCRAPE_BOOK_PAGES = True

# Identify the scraper honestly in the request header
HEADERS = {"User-Agent": "Student data science project (Richfield DS500 assignment)"}

### Being a polite scraper
Before scraping I check `robots.txt`. I also wait a short time between requests, set a timeout, and catch errors so one failed page doesn't stop the whole run.

In [ ]:
robots_response = requests.get(urljoin(BASE_URL, "robots.txt"), headers=HEADERS, timeout=10)
print("Status code for robots.txt:", robots_response.status_code)
if robots_response.status_code == 200:
    print(robots_response.text)
else:
    print("No robots.txt file found, so there are no listed restrictions.")

### A helper function to download a page
`get_soup()` downloads a page with `requests` and turns it into a BeautifulSoup object that we can search. If a request fails, it tries again up to three times.

In [ ]:
def get_soup(url, retries=3):
    """Download a page and return it as a BeautifulSoup object.
    Tries again a few times if the request fails, and returns None if it never works."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()          # raise an error for 404, 500, etc.
            time.sleep(DELAY_SECONDS)            # be polite to the server
            return BeautifulSoup(response.text, "html.parser")
        except requests.exceptions.RequestException as error:
            print(f"Attempt {attempt} failed for {url}: {error}")
            time.sleep(2)
    return None


# Quick test on the home page
home_soup = get_soup(BASE_URL)
print(home_soup.title.text.strip())

### Getting the list of genres
The genre links are inside `<div class="side_categories">`. The first link is "Books" (everything), so I use the CSS selector `ul li ul li a` to get only the genres underneath it.

In [ ]:
def get_categories(soup):
    """Return a dictionary of {category name: category page URL} from the sidebar."""
    categories = {}
    sidebar_links = soup.select("div.side_categories ul li ul li a")
    for link in sidebar_links:
        name = link.text.strip()
        categories[name] = urljoin(BASE_URL, link["href"])
    return categories


categories = get_categories(home_soup)
print("Number of categories found:", len(categories))
print(list(categories.keys())[:10])

### Scraping the listing pages
Each book on a listing page is inside an `<article class="product_pod">` tag:
- the **full title** is in the `title` attribute of the link inside `<h3>` (the visible link text is cut short);
- the **price** is in `<p class="price_color">`;
- the **star rating** is the second class name of `<p class="star-rating Three">`;
- the **stock status** is in `<p class="instock availability">`.

When a genre has more than 20 books, there is an `<li class="next">` link to the next page. The loop keeps following it until it isn't there any more.

In [ ]:
def parse_book_card(card, category, page_url):
    """Pull the fields we need out of one book 'card' on a listing page."""
    title_link = card.h3.a
    return {
        "title": title_link["title"],                      # full title (the link text is shortened)
        "price_raw": card.select_one("p.price_color").text,
        "rating_raw": card.select_one("p.star-rating")["class"][1],   # e.g. "Three"
        "availability_raw": card.select_one("p.availability").text,
        "category": category,
        "book_url": urljoin(page_url, title_link["href"]),
    }


def scrape_category(category, first_page_url):
    """Scrape every page of one category by following the 'next' button."""
    books = []
    page_url = first_page_url
    while page_url is not None:
        soup = get_soup(page_url)
        if soup is None:
            print("Skipping a page that could not be loaded:", page_url)
            break
        for card in soup.select("article.product_pod"):
            books.append(parse_book_card(card, category, page_url))
        next_button = soup.select_one("li.next a")
        if next_button is not None:
            page_url = urljoin(page_url, next_button["href"])
        else:
            page_url = None
    return books

In [ ]:
all_books = []
for number, (category, url) in enumerate(categories.items(), start=1):
    category_books = scrape_category(category, url)
    all_books.extend(category_books)
    print(f"{number:>2}. {category:<20} {len(category_books):>3} books")

print("\nTotal books collected from listing pages:", len(all_books))

### Visiting each book's page
The listing only says "In stock". The book's own page has a *Product Information* table that shows the **number of copies available**, plus the UPC (a unique product code) and the number of reviews. The page also has the book's description.

In [ ]:
def scrape_book_page(book_url):
    """Visit one book's own page for the extra details that the listing does not show."""
    soup = get_soup(book_url)
    if soup is None:
        return {"availability_detail": None, "upc": None,
                "num_reviews_raw": None, "description": None}

    # The 'Product Information' table has rows like: UPC | a897fe39b1053632
    product_info = {}
    for row in soup.select("table.table-striped tr"):
        product_info[row.th.text.strip()] = row.td.text.strip()

    # The description is the paragraph straight after the 'product_description' heading
    description_tag = soup.select_one("#product_description ~ p")
    description = description_tag.text if description_tag is not None else None

    return {
        "availability_detail": product_info.get("Availability"),   # e.g. "In stock (22 available)"
        "upc": product_info.get("UPC"),
        "num_reviews_raw": product_info.get("Number of reviews"),
        "description": description,
    }


if SCRAPE_BOOK_PAGES:
    start_time = time.time()
    for number, book in enumerate(all_books, start=1):
        book.update(scrape_book_page(book["book_url"]))
        if number % 100 == 0:
            print(f"{number} book pages done ({time.time() - start_time:.0f} seconds)")
    print("Finished visiting book pages.")

### Saving the raw data
I save the data exactly as it was scraped, before any cleaning. That way I can rerun the cleaning later without scraping the site again, and I can always check the original values.

In [ ]:
raw_df = pd.DataFrame(all_books)
raw_df.to_csv("books_raw.csv", index=False)
print("Raw data shape:", raw_df.shape)
raw_df.head()

## Step 3 – Explore and clean the data

The raw data is all text, for example `"£51.77"`, `"Three"` and `"In stock (22 available)"`. Before we can analyse it, it needs to be turned into clean numbers and tidy text. In tidy data, every row is one book, every column is one variable, and every cell holds one value.

In [ ]:
# Start the cleaning from the saved raw file, so this part can be re-run
# without scraping the website again
books = pd.read_csv("books_raw.csv")
print(books.shape)
books.info()

### First look at the problems in the raw data

In [ ]:
print("Missing values in each column:")
print(books.isnull().sum())

print("\nExamples of the raw text we need to clean:")
print(repr(books.loc[0, "price_raw"]))
print(repr(books.loc[0, "availability_raw"]))
print(repr(books.loc[0, "availability_detail"]))
print(books["rating_raw"].unique())

### Duplicates
If a book appeared twice, it would count double in the averages. I check exact duplicate rows and repeated book URLs. I also check repeated titles, because two *different* books (different UPC and price) can have the same title, and those should be kept.

In [ ]:
# Exact duplicate rows (the same book scraped twice)
print("Exact duplicate rows:", books.duplicated().sum())

# The book page URL is unique for every product, so it is the safest way to spot repeats
print("Repeated book URLs:", books["book_url"].duplicated().sum())

# Some different books can share the same title, so I check titles separately
repeated_titles = books[books["title"].duplicated(keep=False)].sort_values("title")
print("Rows that share a title with another row:", len(repeated_titles))
repeated_titles[["title", "category", "price_raw", "upc"]]

In [ ]:
rows_before = len(books)
books = books.drop_duplicates(subset="book_url").reset_index(drop=True)
print("Rows removed:", rows_before - len(books))
print("Rows left:", len(books))

**What this shows:** _[After running the cell above, write 2–3 sentences here. How many duplicates were there? Were there any books that share a title but are different products?]_

### Cleaning the price
I remove everything that isn't a digit or a decimal point, then convert the result to a number. This also deals with the `Â£` problem: if the page's text encoding is read wrongly, the £ sign can come through as `Â£`.

In [ ]:
def clean_price(price_text):
    """Turn text like '£51.77' (or a badly encoded 'Â£51.77') into the number 51.77."""
    if pd.isnull(price_text):
        return np.nan
    # Keep only digits and the decimal point
    number_text = re.sub(r"[^0-9.]", "", str(price_text))
    if number_text == "":
        return np.nan
    return float(number_text)


books["price_gbp"] = books["price_raw"].apply(clean_price)
books[["price_raw", "price_gbp"]].head()

### Turning star ratings into numbers
The rating is stored as a word. I use a dictionary to convert it to a number from 1 to 5 so it can be used in calculations and in the model.

In [ ]:
# A dictionary that maps the rating words to numbers
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

books["rating"] = books["rating_raw"].str.strip().str.title().map(rating_map)

print("Ratings that could not be converted:", books["rating"].isnull().sum())
books[["rating_raw", "rating"]].drop_duplicates().sort_values("rating")

### Stock status and stock quantity
- `stock_status`: the text "In stock" or "Out of stock", with the extra spaces and line breaks removed.
- `in_stock`: 1 if the book is in stock, 0 if it isn't.
- `stock_quantity`: the number taken from "(22 available)" using a regular expression.

In [ ]:
def get_stock_quantity(availability_text):
    """Find the number in text like 'In stock (22 available)'. Returns NaN if there is none."""
    if pd.isnull(availability_text):
        return np.nan
    match = re.search(r"(\d+) available", str(availability_text))
    if match:
        return int(match.group(1))
    return np.nan


# Stock status from the listing page, e.g. "In stock"
books["stock_status"] = books["availability_raw"].str.strip()
books["in_stock"] = (books["stock_status"] == "In stock").astype(int)

# Stock quantity from the book's own page, e.g. "In stock (22 available)"
if "availability_detail" in books.columns:
    books["stock_quantity"] = books["availability_detail"].apply(get_stock_quantity)
else:
    books["stock_quantity"] = np.nan

books[["availability_raw", "stock_status", "in_stock", "stock_quantity"]].head()

### Tidying the text fields
I remove extra spaces from titles and genres. In descriptions I also remove the "...more" ending and fill in missing descriptions with an empty string. From the text I create simple number features: how many words are in the title and in the description.

In [ ]:
# Titles and categories: remove extra spaces at the start/end and inside the text
books["title"] = books["title"].str.strip().str.replace(r"\s+", " ", regex=True)
books["category"] = books["category"].str.strip()

# Descriptions: some books have none, and many end with "...more"
books["description"] = (books["description"].fillna("")
                        .str.replace("...more", "", regex=False)
                        .str.replace(r"\s+", " ", regex=True)
                        .str.strip())

# Simple numeric features made from the text fields
books["title_length"] = books["title"].str.split().str.len()           # words in the title
books["description_length"] = books["description"].str.split().str.len()  # words in the description
books["has_description"] = (books["description_length"] > 0).astype(int)

# Number of reviews comes through as text, so turn it into a number
books["num_reviews"] = pd.to_numeric(books["num_reviews_raw"], errors="coerce")

print("Books with no description:", (books["has_description"] == 0).sum())
print("Different review counts found:", books["num_reviews"].unique())

### Handling missing values

In [ ]:
print("Missing values after cleaning:")
print(books[["title", "price_gbp", "rating", "stock_status", "stock_quantity", "category"]].isnull().sum())

In [ ]:
# Rows without a usable price or rating can't be analysed, so they are removed.
rows_before = len(books)
books = books.dropna(subset=["price_gbp", "rating"])
print("Rows removed for missing price or rating:", rows_before - len(books))

# If a book page failed to load, its stock quantity is missing.
# I fill it with the median so one failed page doesn't remove the whole book.
missing_quantity = books["stock_quantity"].isnull().sum()
if missing_quantity > 0:
    books["stock_quantity"] = books["stock_quantity"].fillna(books["stock_quantity"].median())
print("Stock quantities filled with the median:", missing_quantity)

# Store whole numbers as integers
books["rating"] = books["rating"].astype(int)
books["stock_quantity"] = books["stock_quantity"].astype(int)

### Saving the clean dataset
I keep only the cleaned columns, in a sensible order, and save them as **books_clean.csv**. This is the final dataset for the assignment.

In [ ]:
clean_columns = ["upc", "title", "category", "price_gbp", "rating", "stock_status",
                 "in_stock", "stock_quantity", "num_reviews", "title_length",
                 "description_length", "has_description", "description", "book_url"]

books_clean = books[clean_columns].reset_index(drop=True)
books_clean.to_csv("books_clean.csv", index=False)

print("Saved books_clean.csv")
print("Shape (rows, columns):", books_clean.shape)
books_clean.head()

In [ ]:
books_clean.info()
books_clean[["price_gbp", "rating", "stock_quantity", "title_length", "description_length"]].describe().round(2)

**What this shows:** _[After running the cell above, write 2–3 sentences here. Give the minimum, maximum, mean and median price, and the range of stock quantities. Is there anything surprising, e.g. a column where every value is the same?]_

### Visualising the data
Each chart answers one of the questions from Step 1. Every chart has a title and labelled axes.

#### Figure 1 – How are prices spread out?

In [ ]:
plt.hist(books_clean["price_gbp"], bins=30, color="steelblue", edgecolor="white")
plt.axvline(books_clean["price_gbp"].mean(), color="red", linestyle="--",
            label=f"Mean = £{books_clean['price_gbp'].mean():.2f}")
plt.axvline(books_clean["price_gbp"].median(), color="orange", linestyle="--",
            label=f"Median = £{books_clean['price_gbp'].median():.2f}")
plt.title("Figure 1: Distribution of book prices")
plt.xlabel("Price (GBP)")
plt.ylabel("Number of books")
plt.legend()
plt.show()

**What this shows:** _[After running the cell above, write 2–3 sentences here. Is the distribution bell-shaped, skewed, or fairly flat? Are the mean and median close? What is the price range?]_

#### Figure 2 – How many books are there at each rating?

In [ ]:
rating_counts = books_clean["rating"].value_counts().sort_index()

plt.bar(rating_counts.index, rating_counts.values, color="seagreen")
for rating, count in rating_counts.items():
    plt.text(rating, count + 5, str(count), ha="center")
plt.title("Figure 2: Number of books at each star rating")
plt.xlabel("Star rating")
plt.ylabel("Number of books")
plt.xticks([1, 2, 3, 4, 5])
plt.show()

**What this shows:** _[After running the cell above, write 2–3 sentences here. Which rating is most common? Are the ratings spread fairly evenly, or skewed towards low or high ratings?]_

#### Figure 3 – Do higher-rated books cost more?

In [ ]:
average_by_rating = books_clean.groupby("rating")["price_gbp"].mean()
print(average_by_rating.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: average price for each rating
axes[0].bar(average_by_rating.index, average_by_rating.values, color="darkorange")
for rating, price in average_by_rating.items():
    axes[0].text(rating, price + 0.5, f"£{price:.2f}", ha="center")
axes[0].set_title("Figure 3a: Average price by star rating")
axes[0].set_xlabel("Star rating")
axes[0].set_ylabel("Average price (GBP)")
axes[0].set_ylim(0, average_by_rating.max() * 1.2)

# Right: box plot showing the full spread of prices for each rating
sns.boxplot(data=books_clean, x="rating", y="price_gbp", hue="rating",
            palette="Oranges", legend=False, ax=axes[1])
axes[1].set_title("Figure 3b: Spread of prices by star rating")
axes[1].set_xlabel("Star rating")
axes[1].set_ylabel("Price (GBP)")

plt.tight_layout()
plt.show()

**What this shows:** _[After running the cell above, write 2–3 sentences here. Compare the averages. Is there a clear pattern (e.g. a steady increase), or are they all close together? Do the box plots overlap a lot?]_

#### Figure 4 – Which genres have the most books?

In [ ]:
genre_counts = books_clean["category"].value_counts()
print("Number of genres:", len(genre_counts))

top_genres = genre_counts.head(15).sort_values()
plt.figure(figsize=(9, 6))
plt.barh(top_genres.index, top_genres.values, color="slateblue")
plt.title("Figure 4: The 15 genres with the most books")
plt.xlabel("Number of books")
plt.show()

**What this shows:** _[After running the cell above, write 2–3 sentences here. Which genres dominate the catalogue? How many genres have only a handful of books?]_

#### Figure 5 – Do some genres cost more than others?

In [ ]:
# Only genres with at least 10 books, so one or two books don't give a misleading average
genre_summary = (books_clean.groupby("category")
                 .agg(books=("title", "count"), average_price=("price_gbp", "mean"),
                      average_rating=("rating", "mean"))
                 .query("books >= 10")
                 .sort_values("average_price"))
print("Genres with at least 10 books:", len(genre_summary))

plt.figure(figsize=(9, 8))
colours = ["indianred" if price > books_clean["price_gbp"].mean() else "steelblue"
           for price in genre_summary["average_price"]]
plt.barh(genre_summary.index, genre_summary["average_price"], color=colours)
plt.axvline(books_clean["price_gbp"].mean(), color="black", linestyle="--",
            label="Average for whole catalogue")
plt.title("Figure 5: Average price by genre (genres with 10+ books)")
plt.xlabel("Average price (GBP)")
plt.legend(loc="lower right")
plt.show()

genre_summary.round(2)

**What this shows:** _[After running the cell above, write 2–3 sentences here. Name the most and least expensive genres and their averages. How big is the gap? Remember that genres with few books can have unreliable averages.]_

#### Figure 6 – Stock status and stock levels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: stock status counts
status_counts = books_clean["stock_status"].value_counts()
axes[0].bar(status_counts.index, status_counts.values, color="teal")
axes[0].set_title("Figure 6a: Stock status")
axes[0].set_ylabel("Number of books")

# Right: how many copies are in stock
axes[1].hist(books_clean["stock_quantity"], bins=range(0, books_clean["stock_quantity"].max() + 2),
             color="teal", edgecolor="white")
axes[1].set_title("Figure 6b: Number of copies in stock")
axes[1].set_xlabel("Copies available")
axes[1].set_ylabel("Number of books")

plt.tight_layout()
plt.show()

print(status_counts)

**What this shows:** _[After running the cell above, write 2–3 sentences here. Are any books out of stock? What is the most common number of copies?]_

#### Figure 7 – Price compared with stock level and description length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.regplot(data=books_clean, x="stock_quantity", y="price_gbp", ax=axes[0],
            scatter_kws={"alpha": 0.3}, line_kws={"color": "red"})
axes[0].set_title("Figure 7a: Price vs copies in stock")
axes[0].set_xlabel("Copies in stock")
axes[0].set_ylabel("Price (GBP)")

sns.regplot(data=books_clean, x="description_length", y="price_gbp", ax=axes[1],
            scatter_kws={"alpha": 0.3}, line_kws={"color": "red"})
axes[1].set_title("Figure 7b: Price vs description length")
axes[1].set_xlabel("Words in description")
axes[1].set_ylabel("Price (GBP)")

plt.tight_layout()
plt.show()

**What this shows:** _[After running the cell above, write 2–3 sentences here. Is the red trend line flat or sloped? Do the points form a pattern, or are they scattered randomly?]_

#### Figure 8 – Correlation between the numeric variables
Correlation runs from −1 to +1. A value near 0 means there is no straight-line relationship, and a value closer to ±1 means a stronger one.

In [ ]:
numeric_columns = ["price_gbp", "rating", "stock_quantity", "title_length", "description_length"]
correlations = books_clean[numeric_columns].corr()

plt.figure(figsize=(7, 5.5))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Figure 8: Correlation between the numeric variables")
plt.show()

print("Correlation of each variable with price:")
print(correlations["price_gbp"].drop("price_gbp").round(3))

**What this shows:** _[After running the cell above, write 2–3 sentences here. Which variable has the strongest correlation with price? Is any correlation stronger than ±0.1? What does this suggest for the regression model?]_

## Step 4 – Model the data (linear regression)

Linear regression is a **supervised learning** method. It learns from examples where we already know the answer (here, the price) and fits a straight-line equation:

**y = β₀ + β₁x₁ + β₂x₂ + … + βₙxₙ**

- **y** is the price we want to predict;
- **β₀** is the intercept (the starting value);
- each **β** is a coefficient: how much the price changes when that feature goes up by one, with the other features kept the same.

### Choosing the features
I only use information a shop could see *before* setting a price:
- **rating**: the obvious question is whether better-rated books cost more;
- **stock_quantity**: stock levels might be linked to how popular a book is;
- **title_length** and **description_length**: rough measures of how much detail a listing has;
- **genre**: some genres might be priced higher. It has to be turned into 0/1 "dummy" columns because the model only works with numbers.

I leave out `in_stock` and `num_reviews`. Check the cleaning outputs above: if every book has the same value in a column (for example, every book in stock or every book with 0 reviews), that column can't help explain price. The title and description *text* are also left out, because using them would need text-mining methods beyond this course.

In [ ]:
# Numeric features
numeric_features = ["rating", "stock_quantity", "title_length", "description_length"]

# Genre is text, so it is turned into 0/1 "dummy" columns.
# Small genres (under 10 books) are grouped as "Other" so each dummy column has enough books.
genre_counts = books_clean["category"].value_counts()
large_genres = genre_counts[genre_counts >= 10].index
books_clean["genre_group"] = books_clean["category"].where(
    books_clean["category"].isin(large_genres), "Other")

# drop_first=True leaves out one genre to act as the baseline
genre_dummies = pd.get_dummies(books_clean["genre_group"], prefix="genre",
                               drop_first=True, dtype=int)

X = pd.concat([books_clean[numeric_features], genre_dummies], axis=1)
y = books_clean["price_gbp"]

print("Feature table shape:", X.shape)
print("Baseline genre (left out):", sorted(books_clean["genre_group"].unique())[0])
X.head()

### Splitting into training and testing data
The model learns from 80% of the books. I then test it on the other 20%, which it has never seen, to check how well it works on new books. `random_state=42` makes the split the same every time the notebook runs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Testing rows: ", len(X_test))

### Evaluation metrics
- **MAE** (mean absolute error): on average, how many pounds the predictions are off by.
- **MSE** (mean squared error): the average of the squared errors, so big mistakes count more.
- **RMSE** (root mean squared error): the square root of MSE, which puts it back into pounds.
- **R²**: how much of the variation in price the model explains. 1 is perfect, 0 is no better than guessing the average, and below 0 is worse than guessing the average.

In [ ]:
def evaluate(name, actual, predicted):
    """Work out the four evaluation metrics and return them in a dictionary."""
    mse = mean_squared_error(actual, predicted)
    return {"model": name,
            "MAE": mean_absolute_error(actual, predicted),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R2": r2_score(actual, predicted)}

### Baseline: guessing the average price
If a model can't beat this simple guess, it hasn't learned anything useful.

In [ ]:
# Baseline: always guess the average training price.
# A useful model should beat this.
baseline_predictions = np.full(len(y_test), y_train.mean())
results = [evaluate("Baseline (always predict mean)", y_test, baseline_predictions)]
print(f"Baseline guess for every book: £{y_train.mean():.2f}")

### Model 1 – simple linear regression (rating only)

In [ ]:
# Model 1: simple linear regression with one feature (star rating)
simple_model = LinearRegression()
simple_model.fit(X_train[["rating"]], y_train)
simple_predictions = simple_model.predict(X_test[["rating"]])

print(f"Price = {simple_model.intercept_:.2f} + ({simple_model.coef_[0]:.2f} x rating)")
results.append(evaluate("Model 1: rating only", y_test, simple_predictions))

### Model 2 – multiple linear regression (all features)

In [ ]:
# Model 2: multiple linear regression with all the features
multi_model = LinearRegression()
multi_model.fit(X_train, y_train)
multi_predictions = multi_model.predict(X_test)

results.append(evaluate("Model 2: all features", y_test, multi_predictions))

# Training score as well, to check for overfitting
train_r2 = r2_score(y_train, multi_model.predict(X_train))
print(f"Model 2 R² on the training data: {train_r2:.3f}")

### Comparing the models

In [ ]:
results_table = pd.DataFrame(results).set_index("model").round(3)
results_table

**What this shows:** _[After running the cell above, write 2–3 sentences here. Give the MAE, RMSE and R² for each model. Does either model beat the baseline? Is the R² close to 0? Compare Model 2's training R² with its test R²: a big gap means overfitting.]_

### What the coefficients mean

In [ ]:
coefficients = pd.DataFrame({"feature": X.columns, "coefficient": multi_model.coef_})
print(f"Intercept (β₀): {multi_model.intercept_:.2f}")
coefficients.sort_values("coefficient").round(3)

**What this shows:** _[After running the cell above, write 2–3 sentences here. Explain one or two coefficients in plain words, e.g. 'each extra star changes the predicted price by £__'. Are the effects big or small compared with prices that range from about £10 to £60?]_

### Checking the predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, multi_predictions, alpha=0.5)
axes[0].plot([10, 60], [10, 60], color="red", linestyle="--", label="Perfect prediction")
axes[0].set_title("Figure 9a: Actual vs predicted price (test set)")
axes[0].set_xlabel("Actual price (GBP)")
axes[0].set_ylabel("Predicted price (GBP)")
axes[0].legend()

residuals = y_test - multi_predictions
axes[1].hist(residuals, bins=25, color="grey", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Figure 9b: Prediction errors (residuals)")
axes[1].set_xlabel("Actual minus predicted price (GBP)")
axes[1].set_ylabel("Number of books")

plt.tight_layout()
plt.show()

**What this shows:** _[After running the cell above, write 2–3 sentences here. If the model were good, the points would follow the red line. Do they? Are the errors centred on 0, and how wide is their spread?]_

### Limitations of the model
- **Weak relationships:** if none of the features are strongly correlated with price, a straight-line model can't predict price well, whatever settings it uses.
- **Missing information:** real prices depend on things this website doesn't show, such as the author, publisher, page count, format, publication date and demand.
- **Practice data:** Books to Scrape was built for learning to scrape, so its prices and ratings may not behave like those in a real shop.
- **Small genres:** genres with fewer than 10 books were grouped into "Other", so the model can't learn anything specific about them.
- **Linear assumption:** the model assumes each feature affects price in a straight line, which might not be true.

## Step 5 – Communicate the results
_[Write a short summary of your main findings here once the notebook has run: 4–6 bullet points covering price spread, rating vs price, genre vs price, stock, and how well the model did. These go into the report.]_